# Day 2-02｜YOLO Players Footpoint 與 BEV 全自動投影
> Python 籃球運動資料分析課程  
> 本單元直接使用已訓練 YOLO detector 對球場 frame 進行推論，將所有球員 bbox 的 bottom-center footpoint 一次投影到 BEV。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 從模型輸出的 player bbox 取得 bottom-center footpoint。
- 使用 Day 1 Homography 將所有 player footpoint 投影到 BEV。
- 觀察 camera frame 與 BEV 上的對應位置是否合理。


## 課程流程
1. 準備 Day 1 標定好的 Homography pair。
2. 對球場 frame 執行一次 YOLO detection。
3. 將所有 player footpoint 一次投影到 BEV，並觀察結果。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


In [ ]:
HOMOGRAPHY_PAIR_JSON = r"""
{
  "pairs": [
    {"keypoint_name": "right_paint_ft_top", "camera_xy": [818.0, 480.0]},
    {"keypoint_name": "right_free_throw", "camera_xy": [1216.0, 453.0]},
    {"keypoint_name": "right_paint_ft_bottom", "camera_xy": [1410.0, 657.0]},
    {"keypoint_name": "right_paint_baseline_bottom", "camera_xy": [672.0, 623.0]},
    {"keypoint_name": "right_baseline_top", "camera_xy": [674.0, 423.0]},
    {"keypoint_name": "right_baseline_bottom", "camera_xy": [1434.0, 424.0]}
  ]
}
"""


TODO：如果你改用別的球場 frame，請先回 Day 1 重新標定 `HOMOGRAPHY_PAIR_JSON`；不要直接沿用這組 camera 點位。


In [ ]:
import json
import pandas as pd
from src.cv_utils import (
    bottom_center,
    draw_boxes,
    draw_points,
    read_image_rgb,
    save_image_rgb,
    save_json,
    show_image,
    side_by_side,
)
from src.geometry_utils import compute_homography, project_points, render_bev_court
from src.yolo_utils import (
    COURT_KEYPOINT_DISPLAY_NAMES,
    PLAYER_CLASS_NAMES,
    bev_court_bounds,
    court_keypoint_model_path,
    court_vertices_bev_in_bounds,
    detector_model_path,
    draw_court_keypoint_records,
    records_to_dicts,
    run_court_keypoints_on_image,
    run_detector_on_image,
)

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
frame = read_image_rgb(IMAGE_PATH)
bev = render_bev_court(BEV_SPEC_PATH)

bev_points_all = court_vertices_bev_in_bounds(bev_court_bounds(BEV_SPEC_PATH))
bev_lookup = {
    name: [float(x), float(y)]
    for name, (x, y) in zip(COURT_KEYPOINT_DISPLAY_NAMES, bev_points_all)
}

raw_pair_data = json.loads(HOMOGRAPHY_PAIR_JSON)
pairs = []
for raw_pair in raw_pair_data["pairs"]:
    keypoint_name = str(raw_pair["keypoint_name"])
    if "bev_xy" in raw_pair:
        bev_xy = [float(v) for v in raw_pair["bev_xy"]]
    else:
        if keypoint_name not in bev_lookup:
            raise KeyError(f"找不到 BEV keypoint: {keypoint_name}")
        bev_xy = [float(v) for v in bev_lookup[keypoint_name]]
    pairs.append(
        {
            "keypoint_name": keypoint_name,
            "camera_xy": [float(v) for v in raw_pair["camera_xy"]],
            "bev_xy": bev_xy,
        }
    )

H = compute_homography(
    [pair["camera_xy"] for pair in pairs],
    [pair["bev_xy"] for pair in pairs],
)

detections, _ = run_detector_on_image(
    detector_model_path(COURSE_ROOT),
    frame,
    conf=0.25,
    imgsz=960,
)
players = [d for d in detections if d.class_name in PLAYER_CLASS_NAMES]
if not players:
    raise RuntimeError("此 frame 沒有偵測到 player，請調整 conf 或更換影像。")

court_keypoints, _, _ = run_court_keypoints_on_image(
    court_keypoint_model_path(COURSE_ROOT),
    frame,
    bev.shape,
    conf=0.30,
    anchor_confidence=0.45,
    imgsz=960,
    court_bounds=bev_court_bounds(BEV_SPEC_PATH),
)

bboxes = [player.bbox_xyxy for player in players]
labels = [f"P{idx} {player.confidence:.2f}" for idx, player in enumerate(players)]
feet = [bottom_center(bbox) for bbox in bboxes]
bev_feet = project_points(feet, H)
point_labels = [f"P{idx}" for idx in range(len(players))]

frame_base = draw_court_keypoint_records(
    frame,
    court_keypoints,
    show_labels=False,
    point_radius=4,
    line_thickness=2,
)
frame_vis = draw_boxes(frame_base, bboxes, labels, color=(255, 184, 77), thickness=4)
frame_vis = draw_points(frame_vis, feet, point_labels, color=(24, 196, 93), radius=7)

bev_vis = bev.copy()
bev_vis = draw_points(bev_vis, bev_feet, point_labels, color=(255, 99, 72), radius=9)
preview = side_by_side(frame_vis, bev_vis)
show_image(preview, "YOLO players to BEV with court keypose underlay")

out_path = COURSE_ROOT / "assets" / "results" / "d2_02_yolo_players_to_bev.png"
out_json = COURSE_ROOT / "assets" / "results" / "d2_02_yolo_players_to_bev.json"
rows = []
for idx, (player, foot, bev_foot) in enumerate(zip(players, feet, bev_feet)):
    rows.append(
        {
            "index": idx,
            "class_name": player.class_name,
            "confidence": float(player.confidence),
            "bbox_xyxy": [float(v) for v in player.bbox_xyxy],
            "foot_x": float(foot[0]),
            "foot_y": float(foot[1]),
            "bev_x": float(bev_foot[0]),
            "bev_y": float(bev_foot[1]),
        }
    )

save_image_rgb(out_path, preview)
save_json(
    {
        "homography_pairs": pairs,
        "court_keypoints": records_to_dicts(court_keypoints),
        "players": rows,
    },
    out_json,
)
print("players:", len(players))
print("court keypoints:", len(court_keypoints))
print("saved preview:", out_path)
print("saved json:", out_json)
pd.DataFrame(rows)
